# Sampling from a Distribution

This notebook closes Week 1 with a practical superpower: **generating random data that follows any distribution you want**, using only uniform random numbers and the CDF. It is based on the twelfth video of Lesson 2 (Probability Distributions).


## 1. Introduction

You have a dataset — say, heights — but you need it **larger**, and collecting more data is too expensive. Solution: build a distribution from the data you have, then **sample** from it — pick points with the probabilities the distribution dictates. Synthetic data that looks a lot like the original.

Computers only know how to do one random thing well: draw a **uniform** number between 0 and 1. Everything else is built from that.

After completing this notebook you will be able to:

- Sample a discrete distribution by stacking its probabilities into the unit interval.
- Sample **any** distribution (discrete or continuous) by inverting its CDF.
- Explain why uniform samples fed through the inverse CDF reproduce the target distribution.


## 2. Theory

### The strip method (discrete)

A color variable: green with 0.3, blue with 0.5, orange with 0.2. The probabilities sum to 1, so **stack them into a strip** covering $[0, 1]$: green owns $[0, 0.3)$, blue $[0.3, 0.8)$, orange $[0.8, 1]$. Then:

1. generate a uniform number in $[0, 1]$;
2. find which sub-interval it landed in;
3. output that interval's outcome.

Landing in an interval has probability equal to its length — which is exactly the outcome's probability. (Replacing colors with numbers 0, 1, 2 changes nothing.)

### The CDF method (discrete and continuous)

Rotate the stacked bars and push them to the right: the strip's boundaries become the **CDF**. Now the recipe reads:

1. sample $u$ uniformly on the **vertical** interval $[0, 1]$;
2. move horizontally until you hit the CDF curve;
3. read off the value on the **horizontal** axis: $x = F^{-1}(u)$.

For a continuous distribution this is beautiful: picking from a Gaussian directly is hard (areas are hard), but its CDF climbs from 0 to 1 — a uniform sample on the vertical axis maps through the curve to a point on the horizontal axis, and those points follow the Gaussian *exactly*. Where the CDF is **steep** (high density), a wide band of $u$ values maps to a narrow band of $x$ — so many samples land there. Flat CDF regions catch almost nothing.

This is called **inverse transform sampling**.


## 3. Worked Example

For the color distribution, take $u = 0.62$: it exceeds 0.3 (not green), lies in $[0.3, 0.8)$ → **blue**. For a standard normal, $u = 0.62$ maps through $\Phi^{-1}$ to $x \approx 0.31$ — slightly right of center, where the bell is thick.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt

rng = np.random.default_rng(seed=42)

# numpy compatibility: trapezoid was renamed from trapz in numpy 2.0
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz


### The strip method for the color distribution


In [ ]:
outcomes = ["green", "blue", "orange"]
probs = [0.3, 0.5, 0.2]
boundaries = np.cumsum(probs)          # 0.3, 0.8, 1.0

def sample_color(u):
    return outcomes[int(np.searchsorted(boundaries, u, side="right"))]

u = rng.random(100_000)
samples = [sample_color(ui) for ui in u]

for o, p in zip(outcomes, probs):
    freq = samples.count(o) / len(samples)
    print(f"{o:7s} target {p:.2f}   sampled {freq:.4f}")


### Inverse transform for the Gaussian

We build the standard normal CDF on a grid and invert it with interpolation — literally 'find where the horizontal line hits the curve'.


In [ ]:
grid = np.linspace(-5, 5, 4_001)
cdf_grid = np.array([0.5 * (1 + erf(g / sqrt(2))) for g in grid])

def normal_inverse_cdf(u):
    return np.interp(u, cdf_grid, grid)    # horizontal read-off

u = rng.random(200_000)
x = normal_inverse_cdf(u)

print(f"sampled mean:   {x.mean():.4f}  (target 0)")
print(f"sampled spread: {x.std():.4f}  (target 1)")
print(f"P(x <= 1):      {(x <= 1).mean():.4f}  (target 0.8413)")


## 5. Visualizations

### The strip: probabilities stacked into [0, 1]


In [ ]:
fig, ax = plt.subplots(figsize=(9, 2.6))

colors = {"green": "#59a14f", "blue": "#4e79a7", "orange": "#f28e2b"}
left = 0
for o, p in zip(outcomes, probs):
    ax.barh(0, p, left=left, height=0.5, color=colors[o], edgecolor="white")
    ax.text(left + p / 2, 0, f"{o}\n{p}", ha="center", va="center",
            color="white", fontsize=11)
    left += p

u0 = 0.62
ax.annotate(f"u = {u0}", (u0, 0.35), ha="center", fontsize=11,
            arrowprops=dict(arrowstyle="->"), xytext=(u0, 0.75))
ax.set_xlim(0, 1)
ax.set_ylim(-0.5, 1.0)
ax.set_yticks([])
ax.set_xlabel("uniform draw u")
ax.set_title("Stack the probabilities; a uniform draw picks the outcome that owns its interval")
plt.tight_layout()
plt.show()


### The video's centerpiece: uniform verticals through the CDF

Uniform samples on the vertical axis, horizontal read-off through the normal CDF — dense where the CDF is steep, sparse where it's flat.


In [ ]:
def normal_pdf(z):
    return np.exp(-z**2 / 2) / np.sqrt(2 * np.pi)

us = rng.random(12)
xs_hit = normal_inverse_cdf(us)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharex=True)

axes[0].plot(grid, normal_pdf(grid), color="#4e79a7", linewidth=2)
axes[0].plot(xs_hit, np.zeros_like(xs_hit), "v", color="#e15759", markersize=9,
             clip_on=False)
axes[0].set_title("...and these points follow the PDF on the left")
axes[0].set_xlabel("x")
axes[0].set_ylabel("f(x)")
axes[0].set_xlim(-4, 4)

axes[1].plot(grid, cdf_grid, color="#e15759", linewidth=2)
for u0, x0 in zip(us, xs_hit):
    axes[1].plot([-5, x0], [u0, u0], color="gray", linewidth=0.8)
    axes[1].plot([x0, x0], [0, u0], color="gray", linewidth=0.8, linestyle=":")
    axes[1].plot(x0, u0, "o", color="#4e79a7", markersize=5)
axes[1].plot(np.full_like(us, -4.7), us, "<", color="#59a14f", markersize=8)
axes[1].set_title("Uniform picks on the vertical axis, read off through the CDF")
axes[1].set_xlabel("x")
axes[1].set_ylabel("F(x) — uniform samples here")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### 200,000 samples: the histogram is the Gaussian


In [ ]:
plt.figure(figsize=(9, 5))
plt.hist(x, bins=100, density=True, color="#9ecae1", edgecolor="white",
         label="inverse-CDF samples")
plt.plot(grid, normal_pdf(grid), color="#e15759", linewidth=2,
         label="standard normal PDF")
plt.xlim(-4, 4)
plt.title("Uniform in, Gaussian out — inverse transform sampling")
plt.xlabel("x")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


## 6. Exercises

### Basic

1. Using the strip method with boundaries (0.3, 0.8, 1.0): which colors do the draws $u = 0.05,\ 0.35,\ 0.95$ produce?
2. Why must the sub-interval lengths equal the outcome probabilities for the method to be correct?

### Intermediate

3. Sample 10,000 values of a fair die using the strip method and verify each face appears about 1/6 of the time.
4. The Uniform($a$, $b$) CDF is $F(x) = \frac{x-a}{b-a}$. Invert it by hand: what is $F^{-1}(u)$? (You already met this formula in the uniform notebook.)

### Advanced

5. Sample from the **call-center distribution** of earlier notebooks (the gamma-shaped PDF on $[0, 5]$): build its CDF numerically, invert with `np.interp`, generate 100,000 samples, and overlay the histogram on the PDF.


## 7. Solutions


In [ ]:
# Exercise 1
print("1. ", [sample_color(v) for v in [0.05, 0.35, 0.95]])

# Exercise 2
print("2.  A uniform draw lands in an interval with probability equal to its "
      "length — so lengths must equal the target probabilities.")

# Exercise 3
die_bounds = np.cumsum([1/6] * 6)
faces = np.searchsorted(die_bounds, rng.random(10_000), side="right") + 1
print("3. ", {f: round((faces == f).mean(), 3) for f in range(1, 7)})

# Exercise 4
print("4.  u = (x-a)/(b-a)  ->  x = a + (b-a)u — the transform from the "
      "uniform notebook's exercise 5.")

# Exercise 5: sample the call-center distribution
xs5 = np.linspace(0, 5, 2_000)
pdf5 = xs5 ** 3 * np.exp(-xs5 / 0.55)
pdf5 /= np.trapezoid(pdf5, xs5)
cdf5 = np.concatenate([[0], np.cumsum((pdf5[1:] + pdf5[:-1]) / 2 * np.diff(xs5))])
cdf5 /= cdf5[-1]

calls = np.interp(rng.random(100_000), cdf5, xs5)

plt.figure(figsize=(8, 4))
plt.hist(calls, bins=80, density=True, color="#9ecae1", edgecolor="white",
         label="inverse-CDF samples")
plt.plot(xs5, pdf5, color="#e15759", linewidth=2, label="target PDF")
plt.title("Sampling the call-center distribution")
plt.xlabel("Call duration (min)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
print("5.  Histogram matches the target PDF ✓")


## 8. Conclusion

- Sampling turns a distribution back into data — the key to synthetic datasets and to simulation generally.
- **Strip method** (discrete): stack probabilities into $[0,1]$; a uniform draw picks the owner of its interval.
- **Inverse-CDF method** (any distribution): uniform on the vertical axis, read off through the CDF — steep regions catch many samples, exactly reproducing the density.
- This closes Week 1 of the course. The remaining course items are hands-on: pandas and data-analysis labs, the graded quiz, and a programming assignment building the Bayes spam classifier from Lesson 1.
